# Pertemuan 3 - Data Cleaning

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Kelas:** IF401  
**Program Studi:** PJJ Informatika

## Materi
Missing values, duplikat, outlier, JSON, dan API.


## 1. Load Dataset Housing Dirty


In [ ]:
import pandas as pd
import numpy as np
import requests
from pandas import json_normalize

try:
    !gdown "https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo" -O housing_dirty.csv
    df = pd.read_csv("housing_dirty.csv")
except Exception:
    # Fallback agar notebook tetap bisa dijalankan jika file Google Drive tidak dapat diakses
    np.random.seed(42)
    df = pd.DataFrame({
        "id": range(1, 131),
        "luas_m2": np.random.normal(120, 40, 130),
        "harga_juta": np.random.normal(700, 180, 130),
        "kota": np.random.choice(["jakarta ", "BANDUNG", "surabaya", None], 130),
        "kamar": np.random.choice([2, 3, 4, 5, None], 130),
        "tahun_bangun": np.random.choice(range(1990, 2024), 130),
        "kondisi": np.random.choice(["baik", "Bagus", "sedang", None], 130)
    })
    df.loc[0, "harga_juta"] = 10000
    df.loc[1, "luas_m2"] = -50
    df = pd.concat([df, df.iloc[:5]], ignore_index=True)

print("Shape awal:", df.shape)
display(df.head())


## 2. Eksplorasi Awal


In [ ]:
df.info()
display(df.describe().round(2))
print("Missing values:")
print(df.isnull().sum())
print("Duplikat:", df.duplicated().sum())


## 3. Cleaning Data


In [ ]:
df = df.drop_duplicates()

if "kota" in df.columns:
    df["kota"] = df["kota"].astype(str).str.strip().str.title()
if "kondisi" in df.columns:
    df["kondisi"] = df["kondisi"].astype(str).str.strip().str.lower()

for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

def capping_iqr(data, kolom):
    Q1 = data[kolom].quantile(0.25)
    Q3 = data[kolom].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    data[kolom] = data[kolom].clip(lower, upper)
    return data

for col in ["luas_m2", "harga_juta", "kamar", "tahun_bangun"]:
    if col in df.columns:
        df = capping_iqr(df, col)

print("Total missing:", df.isnull().sum().sum())
print("Total duplikat:", df.duplicated().sum())
display(df.head())


## 4. Export dan API


In [ ]:
df.to_csv("housing_clean.csv", index=False)
print("File housing_clean.csv berhasil dibuat.")

url_api = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url_api, timeout=10)
df_api = json_normalize(response.json(), sep="_")
display(df_api[["id", "name", "email", "address_city"]])


## Kesimpulan

Saya mempelajari proses data cleaning mulai dari missing values, duplikat, outlier, export data, hingga mengambil data API. Temuan utama adalah data mentah harus dibersihkan sebelum dianalisis. Keterbatasannya, metode imputasi dan capping outlier masih sederhana.
